# NULL handling

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-04-null-handling.ipynb)

`NULL` means the value is **unknown**. It is not zero, not an empty string, and not a valid answer. Because the
value is unknown, most comparisons with it are unknown too, and a filter keeps only the rows where the condition is
definitely true. Most NULL bugs in analytics come from forgetting that rule.

This chapter shows how NULL behaves in comparisons, filters, aggregates, sorting and `NOT IN`, and how to handle
it deliberately. It ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW contacts AS
SELECT * FROM VALUES (1, 'a@x.com', 10), (2, NULL, NULL), (3, 'b@x.com', 30)
AS contacts(id, email, score)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW tiny_orders AS
SELECT * FROM VALUES (1), (NULL) AS tiny_orders(customer_id)
""")

## 1. Three possible answers

A comparison in SQL returns **true**, **false** or **unknown**. The third value is the one that surprises people.
`WHERE` keeps only the rows where the condition is true.

In [ ]:
spark.sql("""
SELECT
  (1 = 1)         AS one_equals_one,
  (1 = NULL)      AS one_equals_null,
  (NULL = NULL)   AS null_equals_null,
  (NULL IS NULL)  AS is_null
""").show()

`NULL = NULL` is not true, because two unknown values might not be equal. To ask whether a value is missing, use
`IS NULL`. To compare two values where NULL counts as equal to NULL, use the null-safe operator `<=>`.

## 2. Filtering with NULL

A row whose `email` is NULL does not satisfy `email <> 'a@x.com'`, because the answer is unknown. It disappears from
the result. If you want those rows, say so explicitly.

In [ ]:
spark.sql("SELECT id FROM contacts WHERE email <> 'a@x.com' ORDER BY id").show()

spark.sql("SELECT id FROM contacts WHERE email IS NULL OR email <> 'a@x.com' ORDER BY id").show()

## 3. Aggregates skip NULL

`COUNT(*)` counts rows. `COUNT(column)` counts the non-NULL values in a column. `SUM` and `AVG` ignore NULL inputs,
so the average score is taken over two rows, not three.

In [ ]:
spark.sql("""
SELECT
  COUNT(*)     AS all_rows,
  COUNT(email) AS with_email,
  COUNT(score) AS with_score,
  AVG(score)   AS average_score
FROM contacts
""").show()

## 4. COALESCE and NULLIF

`COALESCE(a, b, ...)` returns the first argument that is not NULL. It is how you supply a default.
`NULLIF(a, b)` returns NULL when `a` equals `b`, which turns a placeholder such as `0` into a real NULL.

In [ ]:
spark.sql("""
SELECT id, COALESCE(score, 0) AS score_or_zero
FROM contacts
ORDER BY id
""").show()

spark.sql("SELECT NULLIF(0, 0) AS placeholder_removed, NULLIF(5, 0) AS kept").show()

## 5. Sorting NULL

In Spark, NULL sorts first in ascending order. Add `NULLS LAST` when you want it at the end, which is usually what a
report needs.

In [ ]:
spark.sql("SELECT id FROM contacts ORDER BY score").show()

spark.sql("SELECT id FROM contacts ORDER BY score NULLS LAST").show()

## 6. The NOT IN trap

`x NOT IN (list)` is true only when `x` differs from **every** item in the list. If the list contains a NULL, that
comparison is unknown, so no row can be proven to pass and the query returns nothing. `NOT EXISTS` compares one row
at a time, so the NULL simply matches nobody.

In [ ]:
spark.sql("SELECT customer_id FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM tiny_orders)").show()

spark.sql("""
SELECT c.customer_id
FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM tiny_orders t WHERE t.customer_id = c.customer_id)
ORDER BY c.customer_id
""").show()

### NULL in expressions

Arithmetic with NULL gives NULL, so one missing value makes the whole sum unknown. String functions behave
differently: `concat` returns NULL if any part is NULL, while `concat_ws` skips the NULL parts.

In [ ]:
spark.sql("""
SELECT 1 + NULL                            AS arithmetic,
       concat('a', NULL)                   AS concat_with_null,
       concat_ws('-', 'a', NULL, 'c')      AS concat_ws_skips_null
""").show()

## Interview notes

- **Three-valued logic.** Every comparison with NULL is unknown, and `WHERE` keeps only true rows. Most NULL bugs are
  a filter that silently drops unknown rows.
- **NULL keys in joins** never match, so rows with a NULL key disappear from an inner join. Check the join key's
  NULL count before you join.
- **Sum of nothing is NULL, count of nothing is 0.** Wrap aggregates in `COALESCE` when a report needs zero.

## Summary

- `= NULL` is never true. Use `IS NULL`, or `<=>` for a null-safe comparison.
- `COUNT(column)` and `COUNT(*)` answer different questions.
- Use `COALESCE` for defaults, and `NULLIF` to remove placeholders.
- Prefer `NOT EXISTS` to `NOT IN` when the other side can contain NULL.

## Practice questions

1. How many contacts have no email?
2. What is the average score, ignoring contacts without a score?
3. Which contact ids have an email that is not `'a@x.com'`, including contacts whose email is unknown? Sort by id.
4. Which customers do not appear in `tiny_orders`? Use `NOT EXISTS`, sorted by `customer_id`.

## Answers

### Question 1

In [ ]:
q = "SELECT COUNT(*) AS missing_email FROM contacts WHERE email IS NULL"
spark.sql(q).show()
assert spark.sql(q).collect() == [(1,)]

### Question 2

In [ ]:
q = "SELECT AVG(score) AS average_score FROM contacts"
spark.sql(q).show()
assert spark.sql(q).collect() == [(20.0,)]

### Question 3

In [ ]:
q = "SELECT id FROM contacts WHERE email IS NULL OR email <> 'a@x.com' ORDER BY id"
spark.sql(q).show()
assert spark.sql(q).collect() == [(2,), (3,)]

### Question 4

In [ ]:
q = """
SELECT c.customer_id
FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM tiny_orders t WHERE t.customer_id = c.customer_id)
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2,), (3,), (4,), (5,), (6,), (7,), (8,)]